# UNET 影像處理 

In [9]:
from oxford_pet import OxfordPetDataset
from utils import dice_loss, hybrid_loss
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU 型號: {torch.cuda.get_device_name(0)}")


GPU 型號: NVIDIA GeForce RTX 3050


# 資料預處理


In [10]:

# 使用方式 (不會爆記憶體)
# LOAD 資料，格式為 (影像pixel rgb, mask)
train_ds = OxfordPetDataset('dataset/oxford-iiit-pet', 'dataset/oxford-iiit-pet/train.txt')
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

val_ds = OxfordPetDataset('dataset/oxford-iiit-pet', 'dataset/oxford-iiit-pet/val.txt')
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

test_ds = OxfordPetDataset('dataset/oxford-iiit-pet', 'dataset/oxford-iiit-pet/test_unet.txt', is_test=True)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

# def check_dataset(loader):
#      processed = 0
#      total = len(loader.dataset)
#      for images, masks in loader:
#          processed += images.size(0)   # 每個 batch 的影像數量
#      print(f"處理完成: {processed} / {total} 張影像")

# check_dataset(train_loader)
# check_dataset(val_loader)
# check_dataset(test_loader)


# UNET

In [11]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv(3, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)
        
        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = DoubleConv(128, 64)
        
        self.final = nn.Conv2d(64, 1, kernel_size=1) # 二元分割輸出 1 通道

    def forward(self, x):
        # 收縮與跳躍連接 (Skip Connection)
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))
        s4 = self.enc4(self.pool(s3))
        
        b = self.bottleneck(self.pool(s4))
        
        # 擴張與 Concatenate
        d4 = self.dec4(torch.cat([self.up4(b), s4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), s3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))
        
        return self.final(d1)

# 訓練迴圈
訓練用 BCE當作 loss function，但是驗證時用DICE SCORE

In [13]:
def train_model(train_loader, val_loader, num_epochs=20, lr=1e-4):
    device = "cpu"
    model = UNet().to(device) 
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.BCEWithLogitsLoss()    
    
    best_dice = 0.0
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0
        
        # 訓練階段
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for imgs, masks in pbar:
            imgs, masks = imgs.to(device), masks.to(device)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix(loss=loss.item())

        # 驗證階段 [cite: 187]
        model.eval()
        val_dice = 0
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                outputs = model(imgs)
                val_dice += dice_loss(outputs, masks)
        
        avg_dice = val_dice / len(val_loader)
        print(f"Validation Dice Score: {avg_dice:.4f}")

        # 儲存最佳模型 
        if avg_dice > best_dice:
            best_dice = avg_dice
            torch.save(model.state_dict(), "saved_models/best_unet.pth")
            print(">>> Best model saved!")

    return model

train_model(train_loader, val_loader, num_epochs=1, lr=1e-4)

Epoch 1/1 [Train]:   4%|▍         | 14/324 [03:10<1:10:25, 13.63s/it, loss=0.579]


KeyboardInterrupt: 

# 評估模型

# 輸出結果

In [ ]:

# 假設你已經有 image_id 和 encoded_mask 的 list
image_ids = ["Abyssinian_1", "Abyssinian_2", "Abyssinian_3"]
encoded_masks = ["3 3 8 2", "5 4", ""]

# 建立 DataFrame
df = pd.DataFrame({
    "image_id": image_ids,
    "encoded_mask": encoded_masks
})

print(df)